# BoostARoota Benchmarks

Executable benchmarks comparing BoostARoota vs Boruta vs All Features.

Datasets are registered in `benchmarks/datasets.yaml` and CSVs live in `benchmarks/data/`.
Each cell can be run independently for debugging.

In [1]:
import os
import sys
import yaml
import pandas as pd

# Ensure repo root on path
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'benchmarks' else os.path.abspath(os.getcwd())
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from benchmarks.loaders import load_csv, prepare_features, encode_target
from benchmarks.benchmark import benchmark_dataset

print(f"Repo root: {REPO_ROOT}")

Repo root: /Users/chasedehan/repos/BoostARoota


In [2]:
# Load datasets.yaml - single registration point
config_path = os.path.join(REPO_ROOT, 'benchmarks', 'datasets.yaml')
with open(config_path) as f:
    cfg = yaml.safe_load(f)

datasets = cfg['datasets']
for ds in datasets:
    print(f"- {ds['name']}: {ds.get('display_name', ds['name'])} -> target={ds['target']}, metric={ds['metric']}")

- wine_quality: Wine Quality -> target=quality, metric=mlogloss
- adult: Income -> target=income, metric=logloss
- spambase: Spambase -> target=Class, metric=logloss
- lsvt: LSVT -> target=class, metric=logloss


In [3]:
# Debug single dataset load
def load_dataset(ds_cfg):
    name = ds_cfg['name']
    source = ds_cfg.get('source', {})
    path = source.get('path') or f"benchmarks/data/{name}.csv"
    target = ds_cfg['target']
    sep = source.get('sep', ',')
    X, y = load_csv(path, target, sep=sep)
    X = prepare_features(X)
    y = encode_target(y)
    return X, y

# Test with first dataset
test_ds = datasets[0]
X_test, y_test = load_dataset(test_ds)
print(f"Dataset: {test_ds['name']}")
print(f"X shape: {X_test.shape}, y shape: {y_test.shape}")
print(y_test.value_counts().head())

Dataset: wine_quality
X shape: (6497, 11), y shape: (6497,)
quality
6    2836
5    2138
7    1079
4     216
8     193
Name: count, dtype: int64


In [4]:
# Debug single benchmark run (small sample for speed)
sample_n = 500
X_sample = X_test.sample(n=min(sample_n, len(X_test)), random_state=42)
y_sample = y_test.loc[X_sample.index]

result = benchmark_dataset(
    X_sample, y_sample,
    metric=test_ds.get('metric', 'logloss'),
    task=test_ds.get('task', 'auto'),
    folds=2,
    repeats=1
)
result

{'bar_time': 0.9099804162979126,
 'boruta_time': 4.87526798248291,
 'bar_logloss': 1.2888249158859253,
 'boruta_logloss': 1.3338037729263306,
 'all_logloss': 1.2708896398544312,
 'folds': 2,
 'repeats': 1,
 'task': 'classification'}

In [5]:
# Full benchmark loop over all datasets in datasets.yaml
FOLDS = 5
REPEATS = 1
SAMPLE = None  # set to int for quick debugging, e.g. 500

rows = []
csv_rows = []

for ds_cfg in datasets:
    name = ds_cfg['name']
    display = ds_cfg.get('display_name', name)
    print(f"\n{'='*60}\nBenchmarking {display} ({name})\n{'='*60}")
    
    X, y = load_dataset(ds_cfg)
    if SAMPLE:
        X = X.sample(n=min(SAMPLE, len(X)), random_state=42)
        y = y.loc[X.index]
    
    print(f"Loaded {X.shape[0]} samples, {X.shape[1]} features after encoding")
    
    metric = ds_cfg.get('metric', 'logloss')
    task = ds_cfg.get('task', 'auto')
    
    summary = benchmark_dataset(X, y, metric=metric, task=task, folds=FOLDS, repeats=REPEATS)
    
    bar_time = summary['bar_time']
    boruta_time = summary['boruta_time']
    
    if 'bar_logloss' in summary:
        bar_metric = summary['bar_logloss']
        boruta_metric = summary['boruta_logloss']
        all_metric = summary['all_logloss']
        metric_name = 'LogLoss'
    else:
        bar_metric = summary['bar_rmse']
        boruta_metric = summary['boruta_rmse']
        all_metric = summary['all_rmse']
        metric_name = 'RMSE'
    
    bar_ge_all = 'Yes' if bar_metric <= all_metric else 'No'
    
    print(f"Boruta time: {boruta_time:.3f}s, BAR time: {bar_time:.3f}s")
    print(f"BAR {metric_name}: {bar_metric:.4f}, Boruta: {boruta_metric:.4f}, All: {all_metric:.4f}")
    
    rows.append([
        display,
        ds_cfg.get('target'),
        f"{boruta_time:.3f}s",
        f"{bar_time:.3f}s",
        f"{bar_metric:.4f}",
        f"{boruta_metric:.4f}",
        f"{all_metric:.4f}",
        bar_ge_all
    ])
    csv_rows.append({
        'dataset': name,
        'display_name': display,
        'target': ds_cfg.get('target'),
        'boruta_time': boruta_time,
        'bar_time': bar_time,
        'bar_metric': bar_metric,
        'boruta_metric': boruta_metric,
        'all_metric': all_metric,
        'bar_ge_all': bar_ge_all
    })


Benchmarking Wine Quality (wine_quality)
Loaded 6497 samples, 11 features after encoding
Boruta time: 0.728s, BAR time: 1.336s
BAR LogLoss: 1.0195, Boruta: 1.0193, All: 1.0193

Benchmarking Income (adult)
Loaded 48842 samples, 108 features after encoding
Boruta time: 39.117s, BAR time: 2.491s
BAR LogLoss: 0.2991, Boruta: 0.3000, All: 0.2991

Benchmarking Spambase (spambase)
Loaded 4601 samples, 57 features after encoding
Boruta time: 14.089s, BAR time: 0.862s
BAR LogLoss: 0.1780, Boruta: 0.1779, All: 0.1775

Benchmarking LSVT (lsvt)
Loaded 126 samples, 310 features after encoding
Boruta time: 10.053s, BAR time: 1.241s
BAR LogLoss: 0.3907, Boruta: 0.3853, All: 0.4047


In [6]:
# Output benchmark table (matches README format) - rendered inline
def format_table(rows):
    headers = ['Data Set', 'Target', 'Boruta Time', 'BoostARoota Time',
               'BoostARoota LogLoss', 'Boruta LogLoss', 'All Features LogLoss', 'BAR >= All']
    col_widths = [len(h) for h in headers]
    for r in rows:
        for i, v in enumerate(r):
            col_widths[i] = max(col_widths[i], len(str(v)))
    def fmt(vals):
        return '| ' + ' | '.join(str(v).ljust(col_widths[i]) for i, v in enumerate(vals)) + ' |'
    sep = '|' + '|'.join('-' * (w + 2) for w in col_widths) + '|'
    lines = [fmt(headers), sep] + [fmt(r) for r in rows]
    return '\n'.join(lines)

table = format_table(rows)
print("\n" + "="*60)
print("Benchmark Results")
print("="*60 + "\n")
print(table)

# Display as DataFrame for easy inspection
pd.DataFrame(csv_rows)


Benchmark Results

| Data Set     | Target  | Boruta Time | BoostARoota Time | BoostARoota LogLoss | Boruta LogLoss | All Features LogLoss | BAR >= All |
|--------------|---------|-------------|------------------|---------------------|----------------|----------------------|------------|
| Wine Quality | quality | 0.728s      | 1.336s           | 1.0195              | 1.0193         | 1.0193               | No         |
| Income       | income  | 39.117s     | 2.491s           | 0.2991              | 0.3000         | 0.2991               | No         |
| Spambase     | Class   | 14.089s     | 0.862s           | 0.1780              | 0.1779         | 0.1775               | No         |
| LSVT         | class   | 10.053s     | 1.241s           | 0.3907              | 0.3853         | 0.4047               | Yes        |


,dataset,display_name,target,boruta_time,bar_time,bar_metric,boruta_metric,all_metric,bar_ge_all
0,wine_quality,Wine Quality,quality,0.727562,1.336368,1.019454,1.019317,1.019317,No
1,adult,Income,income,39.116935,2.490963,0.299141,0.299953,0.299114,No
2,spambase,Spambase,Class,14.089293,0.862004,0.178007,0.177922,0.177532,No
3,lsvt,LSVT,class,10.052960,1.240683,0.390653,0.385332,0.404706,Yes
